# Conversation memory and summarization

## Goal

Compare independent calls with thread-scoped history and understand where summarization fits.

## Setup

Select the **Python (LangChain OpenAI)** kernel and run cells from top to bottom.
`OPENAI_API_KEY` and `OPENAI_MODEL` are read only from this course's root `.env`.
Shared settings are in `lesson_support.py`: up to 1024 output tokens, a 30-second
HTTP timeout, no HTTP retries, and LangSmith tracing disabled.
`make_model("another-api-model-id")` creates a different model with the same key.
The default text settings are intended for `gpt-5.6-luna`.
Creating a model does not call the API; `invoke` and consuming `stream` make paid requests.


In [ ]:
from pathlib import Path
import sys

# Find this course's root, whether Jupyter starts at the root or in notebooks/.
current_dir = Path.cwd().resolve()
course_root = next(
    (folder for folder in (current_dir, *current_dir.parents)
     if (folder / "course.json").is_file() and (folder / "lesson_support.py").is_file()),
    None,
)
if course_root is None:
    raise RuntimeError("Start Jupyter from the course root, as described in README.md.")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from lesson_support import make_model

model = make_model()
print("Model:", model.model_name)


## No memory

In [ ]:
from langchain.agents import create_agent


agent = create_agent(
    model=model
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Seán and my favourite colour is green")

response = agent.invoke(
    {"messages": [question]} 
)

In [ ]:
from pprint import pprint

pprint(response)

In [ ]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]} 
)

pprint(response)

## Memory

InMemorySaver stores conversation state in the current Python process (the notebook kernel).
The same thread_id continues a conversation; a different ID starts another one.
Restarting the kernel, or creating a new InMemorySaver, loses this in-memory history.
It does not automatically search previous threads or extract a lasting user profile.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver  


agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Seán and my favourite colour is green")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [ ]:
pprint(response)

In [ ]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

pprint(response)

## Context summarization

The checkpointer stores conversation state. `SummarizationMiddleware` is a separate
component that replaces older messages with a summary while retaining recent messages.
[Documentation](https://docs.langchain.com/oss/python/langchain/middleware/built-in#summarization).

This is a reference snippet, not an executed cell:

```python
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,                 # Model used to generate summaries.
            trigger=("tokens", 4000),     # Approximate token threshold.
            keep=("messages", 10),        # Retain recent messages.
        )
    ],
)
```

After summarization, the model sees the summary plus retained recent messages,
including the current message. Message boundaries may be adjusted to preserve tool-call pairs.
Thresholds can use tokens, message counts, or a fraction of the context window.
Generating a summary requires additional model calls and can lose details.
This reduces active context; it does not automatically erase old database checkpoints.
